In [50]:
import requests
import json
import random
import time
import os

# Create a session to reuse the connection (more browser-like)
session = requests.Session()

def fetch_cosing_data_resilient(page_number=1, page_size=100, max_retries=5):
    url = "https://api.tech.ec.europa.eu/search-api/prod/rest/search"
    params = {
        "apiKey": "285a77fd-1257-4271-8507-f0c6b2961203",
        "text": "*",
        "pageSize": page_size,
        "pageNumber": page_number
    }
    query_json = {
        "bool": {
            "must": [
                {"text": {"query": "*", "fields": ["inciName.exact", "inciUsaName", "innName.exact", "phEurName", "chemicalName", "chemicalDescription"], "defaultOperator": "AND"}},
                {"term": {"itemType": "ingredient"}}
            ]
        }
    }
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:147.0) Gecko/20100101 Firefox/147.0",
        "Origin": "https://ec.europa.eu",
        "Referer": "https://ec.europa.eu/",
        "Accept": "application/json, text/plain, */*"
    }
    files = {'query': (None, json.dumps(query_json), 'application/json')}

    for attempt in range(max_retries):
        try:
            # Using session.post instead of requests.post
            response = session.post(url, params=params, headers=headers, files=files, timeout=30)
            response.raise_for_status()
            return response.json()
        except (requests.exceptions.RequestException, ConnectionResetError) as e:
            # Exponential backoff: 2min, 4min, 8min...
            wait = (attempt + 1) * 120 
            print(f"\n[!] Error on page {page_number}: {e}. Retrying in {wait}s...")
            time.sleep(wait)
            
    return None

# --- Main Scrape Logic ---

file_path = "cosing_full_export.json"
# Load existing progress if the file exists
if os.path.exists(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        all_results = json.load(f)
    # Calculate which page to start on based on existing data
    start_page = (len(all_results) // 100) + 1 
    print(f"Resuming from page {start_page} ({len(all_results)} records already saved).")
else:
    all_results = []
    start_page = 1

page = start_page
page_size = 100 # Keep page size moderate to avoid triggering "heavy query" flags
total_count = 32686

print(f"Starting polite scrape with 20-30s delay...")

while True:
    data = fetch_cosing_data_resilient(page_number=page, page_size=page_size)
    
    if not data or 'results' not in data or not data['results']:
        print(f"\nStopped at page {page}. Check if limit was reached.")
        break
    
    all_results.extend(data['results'])
    
    # Save checkpoint after every successful page
    with open(file_path, 'w', encoding='utf-8') as f:
        json.dump(all_results, f, ensure_ascii=False, indent=4)
        
    print(f"Synced page {page}. Total: {len(all_results)}/{total_count} saved.", end="\r")
    
    if len(all_results) >= data.get('totalResults', total_count):
        print("\nScrape complete!")
        break
        
    page += 1
    
    # Highly polite wait as requested
    time.sleep(random.uniform(20.0, 30.0))

Resuming from page 75 (7400 records already saved).
Starting polite scrape with 20-30s delay...
Synced page 98. Total: 9800/32686 saved.
[!] Error on page 99: ('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None)). Retrying in 120s...
Synced page 100. Total: 10000/32686 saved.
Stopped at page 101. Check if limit was reached.


In [73]:
import requests
import json
import random
import time
import os
import itertools
import string

# --- Configuration ---
RAW_DATA_DIR = "cosing_raw_shards"
STATE_PATH = "scrape_state.json"
session = requests.Session()

# Create the storage directory if it doesn't exist
if not os.path.exists(RAW_DATA_DIR):
    os.makedirs(RAW_DATA_DIR)

# Using 2-letter combinations as per your recent code snippet
chars = string.ascii_lowercase
shards = [''.join(i) for i in itertools.product(chars, repeat=2)]

def load_state():
    if os.path.exists(STATE_PATH):
        with open(STATE_PATH, 'r', encoding='utf-8') as f:
            state = json.load(f)
            return state.get("last_shard_index", -1)
    return -1

def save_state(shard_index):
    with open(STATE_PATH, 'w', encoding='utf-8') as f:
        json.dump({"last_shard_index": shard_index, "shard_value": shards[shard_index]}, f)

def fetch_sharded_data(search_term, page_number=1):
    url = "https://api.tech.ec.europa.eu/search-api/prod/rest/search"
    params = {
        "apiKey": "285a77fd-1257-4271-8507-f0c6b2961203",
        "text": f"*{search_term}*",
        "pageSize": 100,
        "pageNumber": page_number
    }
    
    # Query logic with wildcards enabled
    query_json = {
        "bool": {
            "must": [
                {
                    "text": {
                        "query": f"*{search_term}*", 
                        "fields": ["inciName.exact", "inciUsaName", "chemicalName"], 
                        "defaultOperator": "OR",
                        "analyzeWildcard": True
                    }
                },
                {"term": {"itemType": "ingredient"}}
            ]
        }
    }
    
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:147.0) Gecko/20100101 Firefox/147.0",
        "Origin": "https://ec.europa.eu",
        "Referer": "https://ec.europa.eu/",
        "Accept": "application/json, text/plain, */*"
    }
    
    files = {'query': (None, json.dumps(query_json), 'application/json')}

    try:
        response = session.post(url, params=params, headers=headers, files=files, timeout=30)
        if response.status_code == 200:
            return response.json()
        else:
            print(f"\n[!] Server returned {response.status_code} for shard {search_term}")
    except Exception as e:
        print(f"\n[!] Error on shard '{search_term}': {e}")
        time.sleep(60)
    return None

# --- Main Execution ---
last_idx = load_state()
start_idx = last_idx + 1

print(f"Resuming from shard index {start_idx} ('{shards[start_idx]}').")

try:
    for i in range(start_idx, len(shards)):
        current_shard = shards[i]
        page = 1
        
        while True:
            data = fetch_sharded_data(current_shard, page_number=page)
            
            if not data:
                break
            
            # Save the ENTIRE raw response to a unique file
            filename = f"{current_shard}_p{page}.json"
            file_path = os.path.join(RAW_DATA_DIR, filename)
            
            with open(file_path, 'w', encoding='utf-8') as f:
                json.dump(data, f, ensure_ascii=False, indent=4)
            
            # Check if we need to fetch more pages for this shard
            total_results = data.get('totalResults', 0)
            if page * 100 >= total_results or not data.get('results'):
                break
                
            page += 1
            time.sleep(random.uniform(20.0, 30.0))
        
        save_state(i)
        print(f"[{i}/{len(shards)}] Shard: {current_shard} | Total results found: {total_results}", end="\r")
        
        time.sleep(random.uniform(20.0, 30.0))

except KeyboardInterrupt:
    print("\nPaused by user. Restart the script to resume from this shard.")

Resuming from shard index 0 ('aa').
[18/676] Shard: as | Total results found: 2466
[!] Error on shard 'at': ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))
[29/676] Shard: bd | Total results found: 31314
[!] Error on shard 'be': ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))
[59/676] Shard: ch | Total results found: 3339
[!] Error on shard 'ci': ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))
[65/676] Shard: cn | Total results found: 6779
[!] Error on shard 'co': ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))
[77/676] Shard: cz | Total results found: 2423
[!] Error on shard 'da': ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))
[85/676] Shard: dh | Total results found: 1450
[!] Error on shard 'di': ('Connection aborted.', RemoteDisconnected('Remote end closed conn

In [71]:
import requests
import json

# --- TEST SETTINGS ---
test_shard = "aa"  # Change this to the shard that failed
url = "https://api.tech.ec.europa.eu/search-api/prod/rest/search"
params = {
    "apiKey": "285a77fd-1257-4271-8507-f0c6b2961203",
    "text": f"*{test_shard}*",
    "pageSize": 100,
    "pageNumber": 1
}

# Your updated query with analyzeWildcard and OR operator
query_json = {
    "bool": {
        "must": [
            {
                "text": {
                    "query": f"*{test_shard}*", 
                    "fields": ["inciName.exact", "inciUsaName", "chemicalName"], 
                    "defaultOperator": "OR",
                    "analyzeWildcard": True
                }
            },
            {"term": {"itemType": "ingredient"}}
        ]
    }
}

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:147.0) Gecko/20100101 Firefox/147.0",
    "Origin": "https://ec.europa.eu",
    "Referer": "https://ec.europa.eu/",
    "Accept": "application/json, text/plain, */*"
}

files = {'query': (None, json.dumps(query_json), 'application/json')}

# --- EXECUTION ---
print(f"Testing Shard: {test_shard}")
try:
    # Using a clean request (no session) to isolate the issue
    response = requests.post(url, params=params, headers=headers, files=files, timeout=30)
    
    print(f"Status Code: {response.status_code}")
    print(f"Response Headers: {response.headers.get('Content-Type')}")
    
    if response.status_code == 200:
        data = response.json()
        print("Success! Data received.")
        print(f"Total Results for shard: {data.get('totalResults')}")
        if data.get('results'):
            print(f"First result sample: {data['results'][0].get('inciName')}")
        else:
            print("Warning: 200 OK but results list is empty.")
    else:
        print("--- ERROR DETAIL ---")
        print(response.text[:500]) # Print first 500 chars of the error page

except Exception as e:
    print(f"Request failed critically: {e}")

Testing Shard: aa
Status Code: 200
Response Headers: application/json
Success! Data received.
Total Results for shard: 83
First result sample: None


In [69]:
data

{'apiVersion': '2.144',
 'terms': '*ad*',
 'responseTime': 132,
 'totalResults': 1770,
 'pageNumber': 1,
 'pageSize': 100,
 'sort': 'relevance',
 'groupByField': None,
 'queryLanguage': {'language': 'en', 'probability': 0.0},
 'spellingSuggestion': None,
 'bestBets': [],
 'results': [{'apiVersion': '2.144',
   'reference': '525d1a1d-ef42-4015-bf12-f2400ff8a257',
   'url': 'urn:test',
   'title': None,
   'contentType': 'text/plain',
   'language': 'en',
   'databaseLabel': 'Growth Cosmetic Ingredients and Substances',
   'database': 'GROWTH_COSING',
   'summary': '',
   'weight': 2.5998194,
   'groupById': None,
   'content': '',
   'accessRestriction': False,
   'pages': None,
   'checksum': 'E3B0C44298FC1C149AFBF4C8996FB92427AE41E4649B934CA495991B7852B855',
   'metadata': {'nameOfCommonIngredientsGlossary': ['MELALEUCA LEUCADENDRON CAJUPUTI LEAF EXTRACT'],
    'otherRestrictions': [],
    'note': [],
    'refNo': [],
    'sccsOpinionUrls': [],
    'innName': [],
    'other': [],
    